# Student Dropout Prediction: Multi-Model Comparison

This notebook implements and compares four different machine learning classifiers to predict student dropout using stratified 10-fold cross-validation. The approach focuses on experimentation and baseline model performance comparison without hyperparameter tuning.

## Models Compared
- **Logistic Regression** - Linear baseline with feature scaling
- **Decision Tree** - Interpretable non-linear model
- **Random Forest** - Ensemble method for robust predictions
- **Neural Network (MLP)** - Non-linear deep learning approach

## Table of Contents

1. **[Import Libraries & Load Data](#1-import-libraries--load-data)** - Load necessary Python packages and dataset
2. **[Data Preparation](#2-data-preparation)** - Split data and prepare features and target variables for cross-validation
3. **[Model Configuration](#3-model-configuration)** - Configure all four models with balanced class weights
4. **[Cross-Validation Evaluation](#4-cross-validation-evaluation)** - Train and evaluate all models using 10-fold CV
5. **[Model Comparison Dashboard](#5-model-comparison-dashboard)** - Compare performance metrics across models
6. **[Unified ROC Analysis](#6-unified-roc-analysis)** - Visualize all models' ROC curves together
7. **[Feature Importance Comparison](#7-feature-importance-comparison)** - Compare feature importance across models
8. **[Best Model Selection & Comprehensive Analysis](#8-best-model-selection--comprehensive-analysis)** - Select best model, analyze performance, and provide recommendations
9. **[Summary](#summary)** - Key findings and next steps

## Key Benefits of This Comparison
- **Cross-Validation Focus**: All models evaluated with rigorous 10-fold cross-validation methodology
- **Direct Comparison**: Side-by-side performance metrics and visualizations
- **Model Selection**: Data-driven approach to choosing the best model based on CV performance
- **Feature Insights**: Understanding which features are important across different algorithms
- **Unified ROC Curves**: All four models displayed on a single ROC plot for easy comparison

## 1. Import Libraries & Load Data

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    roc_auc_score, roc_curve, make_scorer
)
from sklearn.utils.class_weight import compute_class_weight

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
# Load the dataset
current_user = os.getlogin()
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
df = pd.read_csv(PROCESSED_DIR / 'feature_selection.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nDataset info:")
print(df.info())

## 2. Data Preparation

**Purpose**: Split data into train/validation sets and prepare features and target variables for cross-validation.

In [ ]:
# Specify the target variable
target_col = 'sdo5_degree_drop_out'

# Split data based on the 'set' column
train_df = df[df['set'] == 'train'].copy()
val_df = df[df['set'] == 'val'].copy()

print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")

# Check class distribution
print("\nClass distribution in training set:")
print(train_df[target_col].value_counts(normalize=True))

In [ ]:
# Define target and feature columns
exclude_cols = ['set', target_col]
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"Target column: {target_col}")
print(f"Number of features: {len(feature_cols)}")
print(f"Excluded columns: {exclude_cols}")

# Prepare feature matrices and target vectors
X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target_col].copy()

# Combine train and validation for cross-validation
X_train_val = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_train_val = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

print(f"\nCombined train+val shape for cross-validation: {X_train_val.shape}")

In [ ]:
# Verify data quality (should be no missing values)
missing_counts = X_train_val.isnull().sum().sum()
print(f"Missing values in features: {missing_counts}")
if missing_counts > 0:
    print("⚠️ Warning: Missing values found! Please perform imputation.")
    missing_cols = X_train_val.isnull().sum()[X_train_val.isnull().sum() > 0]
    print("Columns with missing values:")
    print(missing_cols)
else:
    print("✅ Data quality verified: No missing values found.")

## 3. Model Configuration

**Purpose**: Configure all four models with appropriate parameters and balanced class weights.

In [ ]:
# Calculate baseline accuracy
baseline_accuracy = y_train_val.value_counts(normalize=True).max()
print(f"Baseline accuracy (majority class): {baseline_accuracy:.3f}")

In [ ]:
# Configure all models with balanced class weights
models = {
    'Logistic Regression': LogisticRegression(
        random_state=42,
        class_weight='balanced',
        max_iter=1000
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=42,
        class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    ),
    'Neural Network': None  # Will be configured separately due to class weight handling
}

# Configure Neural Network with computed class weights
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train_val),
    y=y_train_val
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

models['Neural Network'] = MLPClassifier(
    random_state=42,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1
)

print("Model configurations:")
for name, model in models.items():
    print(f"- {name}: {type(model).__name__}")

print(f"\nComputed class weights for Neural Network: {class_weight_dict}")

## 4. Cross-Validation Evaluation

**Purpose**: Train and evaluate all models using stratified 10-fold cross-validation.

In [ ]:
# Set up cross-validation strategy
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define scoring metrics
scoring_metrics = {
    'accuracy': 'accuracy',
    'roc_auc': 'roc_auc',
    'f1': 'f1',
    'recall': 'recall'
}

print(f"Cross-validation strategy: {cv_strategy.n_splits}-fold stratified")
print(f"Scoring metrics: {list(scoring_metrics.keys())}")

In [ ]:
# Initialize results storage
cv_results = {}

print("Starting cross-validation for all models...\n")

for model_name, model in models.items():
    print(f"Evaluating {model_name}...")
    start_time = time.time()
    
    # Extract data
    X_for_cv = X_train_val.values
    
    # Perform cross-validation
    cv_scores = cross_validate(
        model, X_for_cv, y_train_val,
        cv=cv_strategy,
        scoring=scoring_metrics,
        n_jobs=-1,
        return_train_score=True
    )
    
    # Store results
    cv_results[model_name] = cv_scores
    
    end_time = time.time()

print("Cross-validation completed for all models!")

## 5. Model Comparison Dashboard

**Purpose**: Create comprehensive visualizations and tables comparing all models' cross-validation performance, including overfitting analysis and stability assessment.

In [ ]:
# Create performance comparison table
comparison_data = []

for model_name in models.keys():
    scores = cv_results[model_name]
    
    comparison_data.append({
        'Model': model_name,
        'Recall (Mean)': f"{scores['test_recall'].mean():.3f}",
        'Recall (Std)': f"{scores['test_recall'].std():.3f}",
        'Recall-Train (Mean)': f"{scores['train_recall'].mean():.3f}",
        'Recall-Train (Std)': f"{scores['train_recall'].std():.3f}",
        'F1 Score (Mean)': f"{scores['test_f1'].mean():.3f}",
        'F1 Score (Std)': f"{scores['test_f1'].std():.3f}",
        'Accuracy (Mean)': f"{scores['test_accuracy'].mean():.3f}",
        'Accuracy (Std)': f"{scores['test_accuracy'].std():.3f}",
        'ROC AUC (Mean)': f"{scores['test_roc_auc'].mean():.3f}",
        'ROC AUC (Std)': f"{scores['test_roc_auc'].std():.3f}",
        'Accuracy vs Baseline': f"{(scores['test_accuracy'].mean() - baseline_accuracy):.3f}",
        'Train-Val Gap': f"{(scores['train_accuracy'].mean() - scores['test_accuracy'].mean()):.3f}"
    })

comparison_df = pd.DataFrame(comparison_data)
print("Model Performance Comparison:")
print("=" * 120)
print(comparison_df.to_string(index=False))
print("=" * 120)

In [ ]:
# Create performance visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

# Collect data for plotting
model_names = list(models.keys())
accuracy_means = [cv_results[name]['test_accuracy'].mean() for name in model_names]
accuracy_stds = [cv_results[name]['test_accuracy'].std() for name in model_names]
roc_auc_means = [cv_results[name]['test_roc_auc'].mean() for name in model_names]
roc_auc_stds = [cv_results[name]['test_roc_auc'].std() for name in model_names]

# Plot 1: Accuracy comparison with error bars
axes[0, 0].bar(model_names, accuracy_means, yerr=accuracy_stds, capsize=5, alpha=0.7)
axes[0, 0].axhline(y=baseline_accuracy, color='red', linestyle='--', 
                  label=f'Baseline: {baseline_accuracy:.3f}')
axes[0, 0].set_title('Cross-Validation Accuracy')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].tick_params(axis='x', rotation=45)

# Plot 2: ROC AUC comparison with error bars
axes[0, 1].bar(model_names, roc_auc_means, yerr=roc_auc_stds, capsize=5, alpha=0.7, color='orange')
axes[0, 1].axhline(y=0.5, color='red', linestyle='--', label='Random: 0.500')
axes[0, 1].set_title('Cross-Validation ROC AUC')
axes[0, 1].set_ylabel('ROC AUC')
axes[0, 1].legend()
axes[0, 1].tick_params(axis='x', rotation=45)

# Plot 3: Accuracy distributions (box plot)
accuracy_data = [cv_results[name]['test_accuracy'] for name in model_names]
bp1 = axes[1, 0].boxplot(accuracy_data, labels=model_names, patch_artist=True)
for patch in bp1['boxes']:
    patch.set_facecolor('lightblue')
axes[1, 0].set_title('Accuracy Distribution Across Folds')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].tick_params(axis='x', rotation=45)

# Plot 4: ROC AUC distributions (box plot)
roc_auc_data = [cv_results[name]['test_roc_auc'] for name in model_names]
bp2 = axes[1, 1].boxplot(roc_auc_data, labels=model_names, patch_artist=True)
for patch in bp2['boxes']:
    patch.set_facecolor('lightcoral')
axes[1, 1].set_title('ROC AUC Distribution Across Folds')
axes[1, 1].set_ylabel('ROC AUC')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Create comprehensive cross-validation performance visualization (similar to test set visualization)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Cross-Validation Performance: All Models Comparison', fontsize=16, fontweight='bold')

model_names_list = list(models.keys())

# Extract cross-validation metrics
cv_accuracies = [cv_results[name]['test_accuracy'].mean() for name in model_names_list]
cv_roc_aucs = [cv_results[name]['test_roc_auc'].mean() for name in model_names_list]
cv_accuracy_stds = [cv_results[name]['test_accuracy'].std() for name in model_names_list]
cv_roc_auc_stds = [cv_results[name]['test_roc_auc'].std() for name in model_names_list]

# Plot 1: CV Accuracy comparison with error bars
bars1 = axes[0, 0].bar(model_names_list, cv_accuracies, yerr=cv_accuracy_stds, capsize=5, alpha=0.8, color='skyblue')
axes[0, 0].axhline(y=baseline_accuracy, color='red', linestyle='--', alpha=0.7, 
                  label=f'Baseline: {baseline_accuracy:.3f}')
axes[0, 0].set_title('Cross-Validation Accuracy', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].tick_params(axis='x', rotation=45)
# Add values on bars
for bar, acc, std in zip(bars1, cv_accuracies, cv_accuracy_stds):
    height = bar.get_height()
    axes[0, 0].text(bar.get_x() + bar.get_width()/2., height + std + 0.005,
                   f'{acc:.3f}±{std:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 2: CV ROC AUC comparison with error bars
bars2 = axes[0, 1].bar(model_names_list, cv_roc_aucs, yerr=cv_roc_auc_stds, capsize=5, alpha=0.8, color='lightcoral')
axes[0, 1].axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='Random: 0.500')
axes[0, 1].set_title('Cross-Validation ROC AUC', fontweight='bold')
axes[0, 1].set_ylabel('ROC AUC')
axes[0, 1].legend()
axes[0, 1].tick_params(axis='x', rotation=45)
# Add values on bars
for bar, auc, std in zip(bars2, cv_roc_aucs, cv_roc_auc_stds):
    height = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height + std + 0.01,
                   f'{auc:.3f}±{std:.3f}', ha='center', va='bottom', fontsize=9)

# Plot 3: Train vs Validation Performance (shows overfitting)
cv_train_accuracies = [cv_results[name]['train_accuracy'].mean() for name in model_names_list]
x_pos = np.arange(len(model_names_list))
width = 0.35

bars3 = axes[1, 0].bar(x_pos - width/2, cv_train_accuracies, width, label='Training', alpha=0.8, color='steelblue')
bars4 = axes[1, 0].bar(x_pos + width/2, cv_accuracies, width, label='Validation', alpha=0.8, color='darkorange')

axes[1, 0].set_title('Training vs Validation Accuracy (Overfitting Check)', fontweight='bold')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels(model_names_list, rotation=45)
axes[1, 0].legend()

# Add gap annotations
for i, (train_acc, val_acc) in enumerate(zip(cv_train_accuracies, cv_accuracies)):
    gap = train_acc - val_acc
    axes[1, 0].text(i, max(train_acc, val_acc) + 0.02, f'Gap: {gap:.3f}', 
                   ha='center', va='bottom', fontsize=9, 
                   color='red' if gap > 0.05 else 'green')

# Plot 4: Performance Stability (Coefficient of Variation)
stability_scores = [(1 / (cv_results[name]['test_accuracy'].std() + 0.001)) for name in model_names_list]
bars5 = axes[1, 1].bar(model_names_list, stability_scores, alpha=0.8, color='lightgreen')
axes[1, 1].set_title('Model Stability (Higher = More Stable)', fontweight='bold')
axes[1, 1].set_ylabel('Stability Score')
axes[1, 1].tick_params(axis='x', rotation=45)

# Add stability ratings
for bar, stability in zip(bars5, stability_scores):
    height = bar.get_height()
    if stability > 100:
        rating = "Excellent"
        color = 'green'
    elif stability > 50:
        rating = "Good"
        color = 'orange'
    else:
        rating = "Poor"
        color = 'red'
    
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height + 2,
                   rating, ha='center', va='bottom', fontsize=9, color=color, fontweight='bold')

plt.tight_layout()
plt.show()

# Print summary insights
print("\nCROSS-VALIDATION PERFORMANCE INSIGHTS:")
print("=" * 50)

for model_name in model_names_list:
    train_acc = cv_results[model_name]['train_accuracy'].mean()
    val_acc = cv_results[model_name]['test_accuracy'].mean()
    val_std = cv_results[model_name]['test_accuracy'].std()
    gap = train_acc - val_acc
    
    # Assess overfitting
    if gap > 0.05:
        overfitting = "⚠️  High overfitting"
    elif gap > 0.02:
        overfitting = "🟡 Moderate overfitting"
    else:
        overfitting = "✅ Good generalization"
    
    # Assess stability
    if val_std < 0.01:
        stability = "✅ Very stable"
    elif val_std < 0.02:
        stability = "🟡 Moderately stable"
    else:
        stability = "⚠️  Unstable"
    
    print(f"\n{model_name}:")
    print(f"  • CV Accuracy: {val_acc:.3f} ± {val_std:.3f}")
    print(f"  • Train-Val Gap: {gap:.3f} ({overfitting})")
    print(f"  • Stability: {stability}")

## 6. Unified ROC Analysis

**Purpose**: Visualize all models' ROC curves on a single plot for direct comparison.

In [ ]:
# Train final models on full train+val set and generate ROC curves
plt.figure(figsize=(12, 10))

# Define colors for each model
colors = ['blue', 'green', 'red', 'purple']
final_models = {}

for i, (model_name, model) in enumerate(models.items()):
    print(f"Training final {model_name} model for ROC curve...")
    
    # Extract final data
    X_final = X_train_val.values
    
    # Train the final model
    model.fit(X_final, y_train_val)
    final_models[model_name] = model
    
    # Get prediction probabilities
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_final)[:, 1]
    else:
        y_proba = model.decision_function(X_final)
    
    # Calculate ROC curve
    fpr, tpr, _ = roc_curve(y_train_val, y_proba)
    roc_auc = roc_auc_score(y_train_val, y_proba)
    
    # Plot ROC curve
    plt.plot(fpr, tpr, color=colors[i], lw=3, 
             label=f'{model_name} (AUC = {roc_auc:.3f})')

# Add diagonal line for random classifier
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', alpha=0.8, 
         label='Random Classifier (AUC = 0.500)')

# Customize plot
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('ROC Curves: All Models Comparison', fontsize=16, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)

# Add text box with interpretation
textstr = '\n'.join([
    'ROC Curve Interpretation:',
    '• Closer to top-left corner = Better',
    '• AUC > 0.7 = Good performance',
    '• AUC > 0.8 = Excellent performance'
])
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.text(0.6, 0.2, textstr, fontsize=10, verticalalignment='top', bbox=props)

plt.tight_layout()
plt.show()

print("\nROC curve analysis completed!")
print("All models trained and ready for further analysis.")

## 7. Feature Importance Comparison

**Purpose**: Compare feature importance/coefficients across all models to understand which features are consistently important.

In [ ]:
# Extract feature importance/coefficients from each model
feature_importance = pd.DataFrame({
    'Feature': feature_cols
})

for model_name, model in final_models.items():
    if model_name == 'Logistic Regression':
        # Get coefficients (absolute values for importance)
        importance = np.abs(model.coef_[0])
        feature_importance[f'{model_name}_Importance'] = importance
        
    elif model_name in ['Decision Tree', 'Random Forest']:
        # Get feature importance
        importance = model.feature_importances_
        feature_importance[f'{model_name}_Importance'] = importance
        
    elif model_name == 'Neural Network':
        # For neural networks, we'll use a simple approximation
        # This is less interpretable but gives some indication
        if hasattr(model, 'coefs_'):
            # Sum of absolute weights from input to first hidden layer
            importance = np.abs(model.coefs_[0]).sum(axis=1)
            feature_importance[f'{model_name}_Importance'] = importance

# Normalize importance scores to 0-1 scale for comparison
importance_cols = [col for col in feature_importance.columns if 'Importance' in col]
for col in importance_cols:
    if col in feature_importance.columns:
        max_val = feature_importance[col].max()
        if max_val > 0:
            feature_importance[col] = feature_importance[col] / max_val

# Calculate average importance across models
feature_importance['Average_Importance'] = feature_importance[importance_cols].mean(axis=1)

# Sort by average importance
feature_importance = feature_importance.sort_values('Average_Importance', ascending=False)

print("Top 15 Most Important Features (Average Across Models):")
print("=" * 80)
top_features = feature_importance.head(15)[['Feature', 'Average_Importance'] + importance_cols]
print(top_features.to_string(index=False, float_format='%.3f'))

In [ ]:
# Visualize feature importance comparison
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle('Feature Importance Comparison Across Models', fontsize=16, fontweight='bold')

# Get top 10 features for visualization
top_10_features = feature_importance.head(10)

# Plot each model's feature importance
model_cols = [col for col in feature_importance.columns if 'Importance' in col and 'Average' not in col]

for i, col in enumerate(model_cols[:4]):  # Limit to 4 models
    row = i // 2
    col_idx = i % 2
    
    # Create horizontal bar plot
    y_pos = np.arange(len(top_10_features))
    axes[row, col_idx].barh(y_pos, top_10_features[col], alpha=0.7)
    axes[row, col_idx].set_yticks(y_pos)
    axes[row, col_idx].set_yticklabels([name[:30] for name in top_10_features['Feature']], fontsize=10)
    axes[row, col_idx].set_xlabel('Normalized Importance', fontsize=11)
    axes[row, col_idx].set_title(col.replace('_Importance', ''), fontsize=12, fontweight='bold')
    axes[row, col_idx].grid(axis='x', alpha=0.3)
    
    # Invert y-axis to show most important at top
    axes[row, col_idx].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Create a heatmap showing feature importance correlation across models
if len(importance_cols) > 1:
    plt.figure(figsize=(12, 8))
    
    # Create correlation matrix of feature importance
    importance_corr = feature_importance[importance_cols].corr()
    
    # Create heatmap
    sns.heatmap(importance_corr, annot=True, cmap='coolwarm', center=0,
                square=True, fmt='.3f', cbar_kws={'label': 'Correlation'})
    plt.title('Feature Importance Correlation Between Models', fontsize=14, fontweight='bold')
    plt.xlabel('Models', fontsize=12)
    plt.ylabel('Models', fontsize=12)
    
    # Adjust labels
    labels = [col.replace('_Importance', '') for col in importance_cols]
    plt.xticks(range(len(labels)), labels, rotation=45)
    plt.yticks(range(len(labels)), labels, rotation=0)
    
    plt.tight_layout()
    plt.show()
    
    print("\nFeature Importance Correlation Analysis:")
    print("=" * 50)
    print("High correlation indicates models agree on feature importance")
    print("Low correlation suggests different models emphasize different features\n")
    
    # Print correlation summary
    avg_corr = importance_corr.values[np.triu_indices_from(importance_corr.values, k=1)].mean()
    print(f"Average correlation between models: {avg_corr:.3f}")

## 8. Best Model Selection & Comprehensive Analysis

**Purpose**: Select the best performing model based on:
- Accuracy
- F1
- ROC-AUC

In [ ]:
# COMPREHENSIVE MODEL COMPARISON AND BEST MODEL SELECTION
print("COMPREHENSIVE MODEL COMPARISON ANALYSIS")
print("=" * 80)
print(f"Analysis Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Dataset: {df.shape[0]:,} students, {len(feature_cols)} features")
print(f"Evaluation Method: Stratified 10-fold cross-validation")

# 1. MODEL RANKINGS AND SELECTION
print("\n1. MODEL RANKINGS BASED ON CROSS-VALIDATION:")
print("-" * 50)

model_rankings_cv = []
for model_name in models.keys():
    scores = cv_results[model_name]
    
    model_rankings_cv.append({
        'Model': model_name,
        'CV_ROC_AUC_Mean': scores['test_roc_auc'].mean(),
        'CV_ROC_AUC_Std': scores['test_roc_auc'].std(),
        'CV_Accuracy_Mean': scores['test_accuracy'].mean(),
        'CV_Accuracy_Std': scores['test_accuracy'].std(),
        'CV_Recall_Mean': scores['test_recall'].mean(),
        'CV_Recall_Std': scores['test_recall'].std(),
        'CV_F1_Mean': scores['test_f1'].mean(),
        'CV_F1_Std': scores['test_f1'].std(),
        'Stability_Score': 1 / (scores['test_roc_auc'].std() + 0.001),
        'Improvement_Over_Baseline': scores['test_accuracy'].mean() - baseline_accuracy,
        # Combined score: average of normalized ROC AUC and Accuracy
        'Combined_Score': (scores['test_accuracy'].mean() + scores['test_f1'].mean() + scores['test_roc_auc'].mean()) / 3
    })

cv_ranking_df = pd.DataFrame(model_rankings_cv)
# Rank by combined score (equal weight to accuracy and ROC AUC), then by ROC AUC, then by accuracy
cv_ranking_df = cv_ranking_df.sort_values(['Combined_Score'], ascending=[False])

print("Rankings based on Combined Score (Average of Accuracy and ROC AUC):")

# Display rankings with performance and stability
for i, row in cv_ranking_df.iterrows():
    rank = list(cv_ranking_df.index).index(i) + 1
    model = row['Model']
    auc = row['CV_ROC_AUC_Mean']
    auc_std = row['CV_ROC_AUC_Std']
    acc = row['CV_Accuracy_Mean']
    acc_std = row['CV_Accuracy_Std']
    recall = row['CV_Recall_Mean']
    recall_std = row['CV_Recall_Std']
    f1 = row['CV_F1_Mean']
    f1_std = row['CV_F1_Std']
    combined_score = row['Combined_Score']
    improvement = row['Improvement_Over_Baseline']
    
    # Stability assessment
    if auc_std < 0.02 and acc_std < 0.02:
        stability = "Very Stable"
        stability_emoji = "✅"
    elif auc_std < 0.04 and acc_std < 0.04:
        stability = "Moderately Stable"
        stability_emoji = "🟡"
    else:
        stability = "Less Stable"
        stability_emoji = "⚠️"
    
    emoji = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉" if rank == 3 else "📊"
    print(f"{emoji} {rank}. {model}:")
    print(f"     Combined Score: {combined_score:.3f}")
    print(f"     ROC AUC: {auc:.3f} (±{auc_std:.3f}) | Accuracy: {acc:.3f} (±{acc_std:.3f})")
    print(f"     Recall: {recall:.3f} (±{recall_std:.3f}) | F1 Score: {f1:.3f} (±{f1_std:.3f})")
    print(f"     Improvement over baseline: +{improvement:.3f} | Stability: {stability_emoji} {stability}")

# Select best model based on combined score
best_model_name = cv_ranking_df.iloc[0]['Model']

print(f"\n🏆 BEST MODEL SELECTED: {best_model_name}")
print(f"   • Combined Score: {cv_ranking_df.iloc[0]['Combined_Score']:.3f}")
print(f"   • CV ROC AUC: {cv_ranking_df.iloc[0]['CV_ROC_AUC_Mean']:.3f} (±{cv_ranking_df.iloc[0]['CV_ROC_AUC_Std']:.3f})")
print(f"   • CV Accuracy: {cv_ranking_df.iloc[0]['CV_Accuracy_Mean']:.3f} (±{cv_ranking_df.iloc[0]['CV_Accuracy_Std']:.3f})")
print(f"   • CV Recall: {cv_ranking_df.iloc[0]['CV_Recall_Mean']:.3f} (±{cv_ranking_df.iloc[0]['CV_Recall_Std']:.3f})")
print(f"   • CV F1 Score: {cv_ranking_df.iloc[0]['CV_F1_Mean']:.3f} (±{cv_ranking_df.iloc[0]['CV_F1_Std']:.3f})")
print(f"   • Improvement over baseline: +{cv_ranking_df.iloc[0]['Improvement_Over_Baseline']:.3f}")

# Store the ranking dataframe for later use
ranking_df = cv_ranking_df.copy()
ranking_df.columns = ['Model', 'ROC_AUC_Mean', 'ROC_AUC_Std', 'Accuracy_Mean', 'Accuracy_Std', 'Recall_Mean', 'Recall_Std', 'F1_Mean', 'F1_Std', 'Stability_Score', 'Improvement_Over_Baseline', 'Combined_Score']

In [ ]:
# 2. DETAILED ANALYSIS OF BEST MODEL
print(f"\n2. DETAILED ANALYSIS OF BEST MODEL: {best_model_name}")
print("-" * 60)

# Get cross-validation performance for comprehensive analysis
best_cv_scores = cv_results[best_model_name]

print("\nPerformance Metrics:")
print(f"  • Validation Accuracy: {best_cv_scores['test_accuracy'].mean():.4f} (±{best_cv_scores['test_accuracy'].std():.4f})")
print(f"  • Training Accuracy: {best_cv_scores['train_accuracy'].mean():.4f} (±{best_cv_scores['train_accuracy'].std():.4f})")
print(f"  • Validation ROC AUC: {best_cv_scores['test_roc_auc'].mean():.4f} (±{best_cv_scores['test_roc_auc'].std():.4f})")
print(f"  • Validation Recall: {best_cv_scores['test_recall'].mean():.4f} (±{best_cv_scores['test_recall'].std():.4f})")
print(f"  • Validation F1 Score: {best_cv_scores['test_f1'].mean():.4f} (±{best_cv_scores['test_f1'].std():.4f})")

# Overfitting and stability analysis
train_accuracy = best_cv_scores['train_accuracy'].mean()
cv_accuracy = best_cv_scores['test_accuracy'].mean()
cv_roc_auc = best_cv_scores['test_roc_auc'].mean()
cv_recall = best_cv_scores['test_recall'].mean()
cv_f1 = best_cv_scores['test_f1'].mean()
accuracy_gap = train_accuracy - cv_accuracy
acc_std = best_cv_scores['test_accuracy'].std()
auc_std = best_cv_scores['test_roc_auc'].std()

print(f"\nModel Quality Assessment:")
print(f"  • Training-Validation gap: {accuracy_gap:+.4f}")

# Assess overfitting
if accuracy_gap < 0.02:
    overfitting = "✅ No overfitting detected"
elif accuracy_gap < 0.05:
    overfitting = "🟡 Slight overfitting"
else:
    overfitting = "⚠️  Potential overfitting"

# Assess stability
if acc_std < 0.02 and auc_std < 0.02:
    stability = "✅ Very stable"
elif acc_std < 0.04 and auc_std < 0.04:
    stability = "🟡 Moderately stable"
else:
    stability = "⚠️  Less stable"

# Performance rating
if cv_roc_auc >= 0.8:
    auc_rating = "Excellent"
elif cv_roc_auc >= 0.7:
    auc_rating = "Good"
elif cv_roc_auc >= 0.6:
    auc_rating = "Fair"
else:
    auc_rating = "Poor"

# Recall assessment for class imbalance
if cv_recall >= 0.8:
    recall_rating = "Excellent"
elif cv_recall >= 0.7:
    recall_rating = "Good"
elif cv_recall >= 0.6:
    recall_rating = "Fair"
else:
    recall_rating = "Poor"

print(f"  • Overfitting: {overfitting}")
print(f"  • Stability: {stability}")
print(f"  • Performance Rating: {auc_rating} (ROC AUC: {cv_roc_auc:.3f})")
print(f"  • Recall Rating: {recall_rating} (Recall: {cv_recall:.3f}) - Important for identifying at-risk students")
print(f"  • Improvement over baseline: {(cv_accuracy - baseline_accuracy)*100:.1f} percentage points")

# Model configuration details
print(f"\nModel Configuration:")
if best_model_name in final_models:
    best_model = final_models[best_model_name]
    print(f"  • Type: {type(best_model).__name__}")
    key_params = ['random_state', 'class_weight', 'n_estimators', 'max_depth', 'max_iter', 'early_stopping']
    params = best_model.get_params()
    relevant_params = {k: v for k, v in params.items() if k in key_params}
    if relevant_params:
        print(f"  • Key Parameters: {', '.join([f'{k}={v}' for k, v in relevant_params.items()])}")

# Store variables for backward compatibility
best_scores = best_cv_scores

In [ ]:
# 3. KEY INSIGHTS AND PATTERNS
print(f"\n3. KEY INSIGHTS AND PATTERNS:")
print("-" * 40)

# Insight 1: Best performing model type
if best_model_name == 'Random Forest':
    insight1 = "Ensemble methods (Random Forest) perform best, indicating complex feature interactions."
elif best_model_name == 'Neural Network':
    insight1 = "Neural networks capture non-linear patterns effectively for this problem."
elif best_model_name == 'Logistic Regression':
    insight1 = "Linear relationships are sufficient; simpler models work well."
else:
    insight1 = "Tree-based models provide good interpretability with solid performance."

print(f"• Model Type: {insight1}")

# Insight 2: Performance level and readiness
if cv_roc_auc >= 0.8:
    performance_level = "excellent"
    actionable = "Model is ready for further validation and potential deployment."
elif cv_roc_auc >= 0.7:
    performance_level = "good"
    actionable = "Model shows promise but could benefit from feature engineering."
else:
    performance_level = "moderate"
    actionable = "Consider collecting additional features or trying different approaches."

print(f"• Performance Level: Model achieves {performance_level} cross-validation performance.")
print(f"• Readiness: {actionable}")

# Insight 3: Feature importance consistency
if len(importance_cols) > 1:
    avg_importance_corr = feature_importance[importance_cols].corr().values[np.triu_indices_from(
        feature_importance[importance_cols].corr().values, k=1)].mean()
    
    if avg_importance_corr > 0.7:
        consistency_insight = "High agreement between models on feature importance suggests robust feature selection."
    elif avg_importance_corr > 0.4:
        consistency_insight = "Moderate agreement between models on feature importance."
    else:
        consistency_insight = "Low agreement suggests different models focus on different aspects of the data."
    
    print(f"• Feature Consistency: {consistency_insight}")

print(f"\n4. TOP PREDICTIVE FEATURES:")
print("-" * 40)
top_5_features = feature_importance.head(5)
for i, (_, row) in enumerate(top_5_features.iterrows(), 1):
    feature_name = row['Feature'][:50]  # Truncate long names
    avg_importance = row['Average_Importance']
    print(f"{i}. {feature_name}: {avg_importance:.3f}")

print(f"\n5. RECOMMENDATIONS AND NEXT STEPS:")
print("-" * 40)

print(f"📋 Immediate Actions:")
if cv_roc_auc >= 0.75:
    print(f"   ✅ Proceed with hyperparameter tuning for {best_model_name}")
    print(f"   ✅ Consider final validation on hold-out test set")
    print(f"   ✅ Prepare for potential deployment pipeline")
else:
    print(f"   ⚠️  Focus on feature engineering before further optimization")
    print(f"   ⚠️  Collect additional features or domain expertise")
    print(f"   ⚠️  Consider alternative modeling approaches")

print(f"\n📊 Model Development Priorities:")
print(f"   • Hyperparameter tuning focus: {best_model_name}")
print(f"   • Use cross-validation for parameter optimization")
print(f"   • Consider ensemble methods if multiple models perform similarly")

print(f"\n🔬 Long-term Improvements:")
print(f"   • Feature engineering based on top predictive features")
print(f"   • Advanced hyperparameter optimization (Bayesian, Grid Search)")
print(f"   • Ensemble of top-performing models")
print(f"   • Additional data collection for underperforming classes")

print(f"\n6. FINAL MODEL RANKINGS:")
print("-" * 40)
print("Complete Rankings by Cross-Validation ROC AUC:")
for i, (_, row) in enumerate(ranking_df.iterrows(), 1):
    model_name = row['Model']
    roc_auc = row['ROC_AUC_Mean']
    accuracy = row['Accuracy_Mean']
    print(f"{i}. {model_name}: {roc_auc:.3f} AUC, {accuracy:.3f} Acc")

print("\n" + "=" * 80)
print("COMPREHENSIVE ANALYSIS COMPLETE")
print(f"🏆 RECOMMENDED MODEL FOR DEPLOYMENT: {best_model_name}")
print("=" * 80)

### Results & conclusions
The performance of logistic regression, random forest and neural network models are quite similar (i.e. combined metric score ranges from 0.69 to 0.67) whereas the decision tree model performs less well (lower combined metric score, and an average accuracy below baseline). From this we can conclude that the decision tree model does not fit the data well. 

Logistic regression has a problem with converging. Work is needed to investigate. 

Both logistic regression and neural networks models seem to rely heavily (high relative feature importance) on the 'Degree Dental Hygiene' feature. I would not expect that a specific degree would have such a high feature importance: to me it seems indicative to overfitting. When tuning a model we need to apply regularization to allow for better generalization. 

## Save data and best model

In [ ]:
import joblib

# Define data to be saved (best model only)
analysis_data = {
    'cv_results': cv_results[best_model_name],
    'model_details': final_models[best_model_name],
    'best_model_name': best_model_name
}
# Save data
joblib.dump(analysis_data, PROCESSED_DIR / 'model_comparison_analysis.pkl')
print("\nModel comparison analysis data saved successfully!")